**Author:** Ashutosh Jayant  
**Project:** India State Competitiveness Index (ISCI)

# 09 - Comparative Analysis

**This notebook answers one research question.**

**Research Question:** How are different groups of states different from each other?

It puts states into standard groups (region, coast, size) and compares the group averages.
It only uses the numbers. The groups are for comparison and are not part of the index.

## Setup

Load the Version 1.0 results and put each state into groups: region, coastal or inland,
and large or small by population.

In [1]:
import sys
from pathlib import Path
import pandas as pd

root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))
from src import scoring

reports_dir = root / "version2" / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

scores = pd.read_csv(root / "results" / "indicator_scores.csv", index_col=0)
dimensions = pd.read_csv(root / "results" / "dimension_scores.csv", index_col=0)
index = pd.read_csv(root / "results" / "competitiveness_index.csv", index_col=0)
master = pd.read_csv(root / "data" / "processed" / "master_features.csv", index_col=0)
ranked = index[index["Rank"].notna()].sort_values("Rank")

REGION = {
    "North": ["Delhi", "Haryana", "Punjab", "Himachal Pradesh", "Uttarakhand",
              "Uttar Pradesh", "Rajasthan", "Jammu & Kashmir", "Chandigarh"],
    "South": ["Andhra Pradesh", "Karnataka", "Kerala", "Tamil Nadu", "Telangana", "Puducherry"],
    "East": ["Bihar", "Jharkhand", "Odisha", "West Bengal"],
    "West": ["Gujarat", "Maharashtra", "Goa"],
    "Central": ["Madhya Pradesh", "Chhattisgarh"],
    "North-East": ["Assam", "Arunachal Pradesh", "Manipur", "Meghalaya", "Mizoram",
                   "Nagaland", "Sikkim", "Tripura"],
    "Islands": ["Andaman & Nicobar Islands"],
}
COASTAL = ["Gujarat", "Maharashtra", "Goa", "Karnataka", "Kerala", "Tamil Nadu",
           "Andhra Pradesh", "Odisha", "West Bengal", "Puducherry", "Andaman & Nicobar Islands"]

region = pd.Series({s: r for r, states in REGION.items() for s in states})
groups = pd.DataFrame(index=ranked.index)
groups["region"] = region.reindex(ranked.index)
groups["coast"] = ["Coastal" if s in COASTAL else "Inland" for s in ranked.index]
pop = master["population"].reindex(ranked.index)
med = pop.median()
groups["size"] = pop.apply(lambda p: "Large" if p >= med else ("Small" if pd.notna(p) else "Unknown"))

print("region:", dict(groups["region"].value_counts()))
print("coast:", dict(groups["coast"].value_counts()))
print("size (median %.0f lakh):" % med, dict(groups["size"].value_counts()))
print("unmapped:", list(groups.index[groups["region"].isna()]))

region: {'North': np.int64(9), 'North-East': np.int64(8), 'South': np.int64(6), 'East': np.int64(4), 'West': np.int64(3), 'Central': np.int64(2), 'Islands': np.int64(1)}
coast: {'Inland': np.int64(22), 'Coastal': np.int64(11)}
size (median 301 lakh): {'Large': np.int64(17), 'Small': np.int64(16)}
unmapped: []


## Top 10 vs Bottom 10

Compare the average score of the ten highest-ranked and ten lowest-ranked states for each
number, and find where the gaps are biggest.

In [2]:
r = index["Rank"]
top = r[(r >= 1) & (r <= 10)].index
bottom = r[(r >= 24) & (r <= 33)].index

comp = pd.DataFrame({"top 10": scores.loc[top].mean(),
                     "bottom 10": scores.loc[bottom].mean()})
comp["gap"] = comp["top 10"] - comp["bottom 10"]
print(comp.round(3).sort_values("gap", ascending=False).to_string())

                     top 10  bottom 10    gap
factory_density       0.727      0.108  0.618
manufacturing_share   0.583      0.128  0.455
investment_rate       0.524      0.132  0.393
ger                   0.634      0.259  0.375
msme_density          0.647      0.311  0.336
td_losses             0.807      0.486  0.321
cd_ratio              0.502      0.295  0.207
unemployment_rate     0.649      0.509  0.140
per_capita_power      0.126      0.025  0.101
life_expectancy       0.599      0.529  0.070
road_density          0.126      0.291 -0.165


## Region comparison

Group states by region and compare the average score, the basic-inputs score, and the
industry score. Some regions have very few states, so their averages are read with care.

In [3]:
df = pd.DataFrame({
    "score": ranked["competitiveness_score"],
    "FC": dimensions.loc[ranked.index, "factor_conditions"],
    "Related": dimensions.loc[ranked.index, "related_supporting"],
    "region": groups["region"],
})
region_avg = df.groupby("region").agg(
    states=("score", "size"),
    avg_score=("score", "mean"),
    avg_FC=("FC", "mean"),
    avg_Related=("Related", "mean"),
).sort_values("avg_score", ascending=False)
print(region_avg.round(3).to_string())

            states  avg_score  avg_FC  avg_Related
region                                            
West             3      0.570   0.482        0.712
South            6      0.510   0.509        0.510
North            9      0.410   0.426        0.386
East             4      0.354   0.376        0.315
Central          2      0.330   0.346        0.303
North-East       8      0.288   0.346        0.199
Islands          1      0.254   0.395        0.112


## Region comparison

Here we group states by region and take the average score of each region. Some regions
have very few states, so their average is not as reliable. West has only 3 states, Central
has 2, and Islands has just 1 (Andaman & Nicobar), so its average is really just one state.

Average score by region, highest to lowest:

- West: 0.57 (3 states)
- South: 0.51 (6 states)
- North: 0.41 (9 states)
- East: 0.35 (4 states)
- Central: 0.33 (2 states)
- North-East: 0.29 (8 states)
- Islands: 0.25 (1 state)

What the numbers show:

The West region has the highest average. This comes mostly from its industry score
(0.71), the highest of all regions. The South is second, and it is balanced: its
basic-inputs score and its industry score are almost the same (both about 0.51). The
North-East has the lowest average among the larger regions, mainly because of its low
industry score (0.20). The Islands group is just one state, so it is not a real regional
average.

## Coastal vs inland

Compare the coastal and inland states on the average score, the two parts, and each
number.

In [4]:
df = pd.DataFrame({
    "score": ranked["competitiveness_score"],
    "FC": dimensions.loc[ranked.index, "factor_conditions"],
    "Related": dimensions.loc[ranked.index, "related_supporting"],
    "coast": groups["coast"],
})
print(df.groupby("coast").agg(
    states=("score", "size"),
    avg_score=("score", "mean"),
    avg_FC=("FC", "mean"),
    avg_Related=("Related", "mean"),
).round(3).to_string())

coastal_idx = groups.index[groups["coast"] == "Coastal"]
inland_idx = groups.index[groups["coast"] == "Inland"]
gap = (scores.loc[coastal_idx].mean() - scores.loc[inland_idx].mean()).sort_values(ascending=False)
print("\nindicator gap (coastal minus inland):")
print(gap.round(3).to_string())

         states  avg_score  avg_FC  avg_Related
coast                                          
Coastal      11      0.484   0.476        0.504
Inland       22      0.353   0.385        0.304

indicator gap (coastal minus inland):
factory_density        0.224
investment_rate        0.216
ger                    0.206
msme_density           0.182
manufacturing_share    0.175
life_expectancy        0.140
td_losses              0.139
cd_ratio               0.103
unemployment_rate      0.092
per_capita_power       0.035
road_density          -0.093


## Coastal vs inland

Here we compare the 11 coastal states with the 22 inland states.

Coastal states have a higher average score (0.48) than inland states (0.35). The gap is
bigger on industry (0.50 against 0.30) than on basic inputs (0.48 against 0.39).

The biggest differences, where coastal states lead, are on the industry and education
numbers:

- Factory density (gap 0.22)
- Investment rate (0.22)
- School enrolment (0.21)
- MSME density (0.18)
- Manufacturing share (0.18)

Inland states are ahead only on road density.

What the numbers show:

Coastal states score higher, and the difference is mostly on the industry numbers. The
data shows this difference but does not tell us why.

## Large vs small states

Split states by population at the middle (median) value, and compare the two groups on the
average score, the two parts, and each number.

In [5]:
df = pd.DataFrame({
    "score": ranked["competitiveness_score"],
    "FC": dimensions.loc[ranked.index, "factor_conditions"],
    "Related": dimensions.loc[ranked.index, "related_supporting"],
    "size": groups["size"],
})
print(df.groupby("size").agg(
    states=("score", "size"),
    avg_score=("score", "mean"),
    avg_FC=("FC", "mean"),
    avg_Related=("Related", "mean"),
).round(3).to_string())

large_idx = groups.index[groups["size"] == "Large"]
small_idx = groups.index[groups["size"] == "Small"]
gap = (scores.loc[large_idx].mean() - scores.loc[small_idx].mean()).sort_values(ascending=False)
print("\nindicator gap (large minus small):")
print(gap.round(3).to_string())

       states  avg_score  avg_FC  avg_Related
size                                         
Large      17      0.431   0.435        0.424
Small      16      0.360   0.393        0.314

indicator gap (large minus small):
investment_rate        0.204
unemployment_rate      0.185
cd_ratio               0.163
td_losses              0.081
msme_density           0.081
factory_density        0.081
manufacturing_share    0.073
per_capita_power       0.003
ger                   -0.057
life_expectancy       -0.073
road_density          -0.182


## Large vs small states

Here we split states by population, using the middle (median) value. This gives 17 large
states and 16 small states.

Large states have a slightly higher average score (0.43) than small states (0.36). The gap
is bigger on industry (0.42 against 0.31) than on basic inputs (0.44 against 0.39).

Where large states lead:

- Investment rate (gap 0.20)
- Low unemployment (0.19)
- Credit-to-deposit ratio (0.16)

Where small states lead:

- Road density (0.18 in their favour)
- Life expectancy (0.07)
- School enrolment (0.06)

What the numbers show:

The size gap is smaller than the coastal gap. Being large helps only a little, mostly on
investment and jobs. Small states do better on road density, which is measured per person,
so states with fewer people can score higher on it.

## Case studies

Compare five chosen pairs of states, one at a time. For each pair, show the ranks and the
numbers where each state leads. A lead counts only if the gap is more than 0.02.

In [6]:
pairs = [("Gujarat", "Maharashtra"), ("Tamil Nadu", "Karnataka"),
         ("Bihar", "Uttar Pradesh"), ("Punjab", "Haryana"), ("Kerala", "Tamil Nadu")]

for a, b in pairs:
    diff = (scores.loc[a] - scores.loc[b]).dropna()
    a_ahead = diff[diff > 0.02].sort_values(ascending=False).head(3)
    b_ahead = (-diff[diff < -0.02]).sort_values(ascending=False).head(3)
    print(f"{a} (rank {int(index.loc[a, 'Rank'])}) vs {b} (rank {int(index.loc[b, 'Rank'])})")
    print("  " + a + " ahead:", ", ".join(f"{k} ({v:.2f})" for k, v in a_ahead.items()))
    print("  " + b + " ahead:", ", ".join(f"{k} ({v:.2f})" for k, v in b_ahead.items()))
    print()

Gujarat (rank 3) vs Maharashtra (rank 8)
  Gujarat ahead: manufacturing_share (0.56), investment_rate (0.55), factory_density (0.52)
  Maharashtra ahead: life_expectancy (0.33), ger (0.33), msme_density (0.24)

Tamil Nadu (rank 2) vs Karnataka (rank 12)
  Tamil Nadu ahead: factory_density (0.61), cd_ratio (0.33), life_expectancy (0.30)
  Karnataka ahead: road_density (0.06), unemployment_rate (0.06)

Bihar (rank 30) vs Uttar Pradesh (rank 27)
  Bihar ahead: life_expectancy (0.35)
  Uttar Pradesh ahead: ger (0.25), td_losses (0.14), factory_density (0.14)

Punjab (rank 7) vs Haryana (rank 10)
  Punjab ahead: life_expectancy (0.29), td_losses (0.13), factory_density (0.13)
  Haryana ahead: unemployment_rate (0.20), cd_ratio (0.10), investment_rate (0.08)

Kerala (rank 15) vs Tamil Nadu (rank 2)
  Kerala ahead: road_density (0.11), life_expectancy (0.11), ger (0.07)
  Tamil Nadu ahead: factory_density (0.63), msme_density (0.37), unemployment_rate (0.37)



## Case studies

Here we compare five pairs of states, one pair at a time. Each pair is chosen for a
reason. For each pair we show the ranks and the numbers where each state leads. A lead is
counted only if the gap is more than 0.02.

### Gujarat vs Maharashtra

Two large western industrial states with different patterns.
Gujarat is rank 3 (0.594). Maharashtra is rank 8 (0.505).
Gujarat leads on manufacturing share (0.56), investment rate (0.55) and factory density
(0.52).
Maharashtra leads on life expectancy (0.33), school enrolment (0.33) and MSME density
(0.24).
So Gujarat is ahead on heavy industry, and Maharashtra is ahead on health, schooling and
small businesses.

### Tamil Nadu vs Karnataka

Two large southern states, but one ranks much higher.
Tamil Nadu is rank 2 (0.605). Karnataka is rank 12 (0.446).
Tamil Nadu leads on factory density (0.61), credit-to-deposit ratio (0.33) and life
expectancy (0.30).
Karnataka leads only on road density (0.06) and unemployment (0.06), and by small amounts.
So Tamil Nadu is ahead on almost everything, especially industry.

### Bihar vs Uttar Pradesh

Two large northern states near the bottom of the ranking.
Bihar is rank 30 (0.245). Uttar Pradesh is rank 27 (0.280).
Bihar leads only on life expectancy (0.35).
Uttar Pradesh leads on school enrolment (0.25), low power loss (0.14) and factory density
(0.14).
So both are weak, but Uttar Pradesh is a little ahead on schooling and industry.

### Punjab vs Haryana

Neighbouring states with similar geography but different results.
Punjab is rank 7 (0.506). Haryana is rank 10 (0.480).
Punjab leads on life expectancy (0.29), low power loss (0.13) and factory density (0.13).
Haryana leads on low unemployment (0.20), credit-to-deposit ratio (0.10) and investment
rate (0.08).
So the two are close, but they are strong on different numbers.

### Kerala vs Tamil Nadu

A high human-development state against a high-industry state.
Kerala is rank 15 (0.417). Tamil Nadu is rank 2 (0.605).
Kerala leads on road density (0.11), life expectancy (0.11) and school enrolment (0.07).
Tamil Nadu leads on factory density (0.63), MSME density (0.37) and low unemployment
(0.37).
So Kerala is ahead on some basic conditions, but Tamil Nadu is far ahead on industry and
jobs.